Build Composite Video by start-time and end-time

In [ ]:
import os
import shutil
import pickle
import subprocess
# import pandas as pd
from utils_mocap_viz.generate_views import (    # organize this
    # get_output_dir,
    prepare_videos
)

from utils_mocap_viz.animated_merged_phase_analysis import animate_merged_phase_analysis, animate_merged_phase_analysis_with_user_window
from utils_dance_anim.dance_dot import animate_dance_phase_analysis, animate_dance_phase_analysis_with_user_window
from utils_pipeline.pipeline_B import *
from utils_composite_video.by_time_segment import *


### Layout

In [ ]:
views_to_generate = ['front']       # skeleton views ['front', 'right' 'left', 'top'] 

layout_5views = [
    # Top row - side by side
    {'view': 'video_mix', 'x': 0, 'y': 0, 'width': 960, 'height': 540},
    {'view': 'front_view', 'x': 960, 'y': 0, 'width': 960, 'height': 540},
    
    # Bottom row - stacked vertically
    {'view': 'joint_Hips', 'x': 0, 'y': 540, 'width': 960, 'height': 270},
    {'view': 'plot', 'x': 0, 'y': 810, 'width': 960, 'height': 270},
    
    # {'view': 'drum_dot', 'x': 960, 'y': 540, 'width': 960, 'height': 540},
    {'view': 'drum_dot', 'x': 960, 'y': 540, 'width': 960, 'height': 270},
    {'view': 'dance_dot', 'x': 960, 'y': 810, 'width': 960, 'height': 270},
]

layout_2views = [
    # Top row 
    {'view': 'video_mix', 'x': 480, 'y': 0, 'width': 960, 'height': 540},
    {'view': 'drum_dot', 'x': 0, 'y': 540, 'width': 1920, 'height': 540}, # bottom row

]

layouts_to_export = {
                    "layout_2views": layout_2views,
                     "layout_5views": layout_5views
                     }

### Config

In [ ]:
with open('data/selected_piece_list.pkl', 'rb') as f:
    piece_list = pickle.load(f)

# for file_name in piece_list:

file_name = "BKO_E1_D1_01_Suku"
traj_dir  = "traj_files_presentation"
status    = "included"   # or "excluded"
traj_threshold = "0.001"        # or any other threshold

bvh_dir = os.path.join("data", "bvh_files")
bvh_file = os.path.join(bvh_dir, file_name + "_T")

# path to onsets and cycles csv files
cycles_csv_path = f"data/virtual_cycles/{file_name}_C.csv"
onsets_csv_path = f"data/drum_onsets/{file_name}.csv"
dance_csv_path = f"data/dance_onsets/{file_name}_T_dance_onsets.csv"

cyc_df = pd.read_csv(cycles_csv_path)
cycle_onsets = cyc_df["Virtual Onset"].values

m_idx = 2
mode = ["group", "individual", "audience"]
dance_mode = mode[m_idx]

motion_data_dir = "data/motion_data_pkl"
dance_mode_path = f"data/dance_modes_ts/{file_name}_{dance_mode}.pkl"

# load dance mode time segments
if os.path.exists(dance_mode_path):
    with open(dance_mode_path, "rb") as f:
        dmode_ts = pickle.load(f)
else:
    print(f"{file_name} {dance_mode} does not exist")
    # continue

# if len(dmode_ts) > 1:
#     print(f"{file_name}", len(dmode_ts))
mode_seg_idx = 3
mode_start_time, mode_end_time = dmode_ts[mode_seg_idx]

    
    

print("Dance Mode:", dance_mode)
print(f"Mode start time: {mode_start_time:.2f}")
print(f"Mode end time: {mode_end_time:.2f}")

#---------------------
# start_time = mode_start_time
# end_time = mode_end_time

# # base_output_dir = os.path.join("composite_videos", dance_mode, f"{file_name}_{start_time:.2f}_{end_time:.2f}")
# # os.makedirs(base_output_dir, exist_ok=True)

# cycle_onsets_in_range = cycle_onsets[(cycle_onsets >= start_time) & (cycle_onsets <= end_time)] 
# traj_tuples = [ (cycle_onsets_in_range[i], cycle_onsets_in_range[i+1], 0) for i in range(len(cycle_onsets_in_range)-1)]
# print("total number of cycles in range:", len(traj_tuples))
    # print("\n")

In [ ]:
# choose start and end time for the composite video

start_time = 162
end_time = 166

# check if start_time and end_time are within the dance mode time segment
if start_time < mode_start_time or end_time > mode_end_time:
    print(f"Start time {start_time:.2f} or end time {end_time:.2f} is outside the dance mode time segment {mode_start_time:.2f} to {mode_end_time:.2f}")
    raise ValueError("Start time or end time is outside the dance mode time segment")


cycle_onsets_in_range = cycle_onsets[(cycle_onsets >= start_time) & (cycle_onsets <= end_time)] 


traj_tuples = [ (cycle_onsets_in_range[i], cycle_onsets_in_range[i+1], 0) for i in range(len(cycle_onsets_in_range)-1)]
traj_tuples


base_output_dir = os.path.join("composite_videos", dance_mode+"1", f"{file_name}_{start_time:.2f}_{end_time:.2f}")
os.makedirs(base_output_dir, exist_ok=True)

traj_tuples

### Generate trajectory Plot + trimmed dance video

In [ ]:
extract_forward_cycle_videos_and_plots(
    file_name = file_name,
    windows = traj_tuples,  # List of (win_start, win_end, t_poi) tuples
    base_path_logs = "data/logs_v4_0.007_foot_jun3",            # logs_v4_0.007_foot_jun3       logs_v2_may
    figsize = (10, 3),
    dpi = 200,
    save_dir = base_output_dir,
    legend_flag = False,
    )

### Generate Skeletal video + trimmed_video_mix + drum dot

In [ ]:
# video_path = os.path.join("data", "videos", f"{file_name}_pre_R_Mix.mp4")

output_dir1 = os.path.join(base_output_dir, "video_skeleton")
os.makedirs(output_dir1, exist_ok=True)

output_dir3 = os.path.join(base_output_dir, "drum_dot_merged")
os.makedirs(output_dir3, exist_ok=True)

output_dir4 = os.path.join(base_output_dir, "dance_dot")
os.makedirs(output_dir4, exist_ok=True)



for c_start_time, c_end_time, _ in traj_tuples:

    # Drum dot plot video
    animate_merged_phase_analysis_with_user_window(
        file_name= file_name,
        W_start= mode_start_time,   # dance mode start time
        W_end= mode_end_time,       # dance mode end time
        user_start= c_start_time,     # user defined start time
        user_end= c_end_time,         # user defined end time
        cycles_csv_path= cycles_csv_path,
        onsets_csv_path= onsets_csv_path,
        save_dir=output_dir3,
        figsize=(10, 3),
        dpi=200,
        legend_flag=False
    )


    # Dance dot plot video
    animate_dance_phase_analysis_with_user_window(
        file_name= file_name,
        W_start= mode_start_time,   # dance mode start time
        W_end= mode_end_time,       # dance mode end time
        user_start= c_start_time,     # user defined start time
        user_end= c_end_time,         # user defined end time
        cycles_csv_path= cycles_csv_path,
        dance_csv_path= dance_csv_path,
        save_dir=output_dir4,
        figsize=(10, 3),
        dpi=200,
    )
    
    
    # Generate Skeleton views
    view_videos = prepare_videos(
        filename= bvh_file,
        start_time= c_start_time,
        end_time= c_end_time,
        views_to_generate = views_to_generate,
        video_path= None,             # video_path, wont generate video
        video_size= (1280, 720),
        fps= 24,
        output_dir = output_dir1
    )
    

### Generate animated kinematic plots

In [ ]:
# Available markers: ['Hips', 'LeftHip', 'LeftKnee', 'LeftAnkle', 'LeftToe', 
# 'LeftToeEnd', 'RightHip', 'RightKnee', 'RightAnkle', 'RightToe', 'RightToeEnd', 
# 'Chest', 'Chest2', 'Chest3', 'Chest4', 'LeftCollar', 'LeftShoulder', 'LeftElbow', 
# 'LeftWrist', 'LeftWristEnd', 'RightCollar', 'RightShoulder', 'RightElbow', 
# 'RightWrist', 'RightWristEnd', 'Neck', 'Head', 'HeadEnd']

mvnx_to_bvh = {
    'x': 'z',  # forward mvnx → backward bvh
    'y': 'x',  # side mvnx → side bvh
    'z': 'y',  # vertical mvnx → vertical bvh
}

joint_name = "Hips"  
axis = 'z'      # z is vertical in mvnx files

output_dir2 = os.path.join(base_output_dir, joint_name)
os.makedirs(output_dir2, exist_ok=True)

extract_kinematic_cycle_plots(
                file_name= file_name,
                windows= traj_tuples,
                joint_name= joint_name,
                axis= mvnx_to_bvh[axis],
                output_dir2= output_dir2,
                figsize = (10, 3),  # 2000 x 600 px
                dpi= 200,
                legend_flag = False,
                )

In [ ]:
# Test
from utils_composite_video.by_time_segment import extract_hand_distance_cycle_plots

output_dir3 = os.path.join(base_output_dir, "hand_distance")
os.makedirs(output_dir3, exist_ok=True)

extract_hand_distance_cycle_plots(
    file_name=file_name,
    windows=traj_tuples,
    axis='y',
    output_dir2=output_dir3,
    use_relative_velocity=True  # Plot distance
)



### Prepare for concatenation

In [ ]:
concatenate_and_overlay_videos(file_name, joint_name, base_output_dir, views_to_generate)        # modify

concat_file_list = [f for f in os.listdir(base_output_dir) if f.lower().endswith(".mp4")]
concat_dict = {
    f.replace('_concat.mp4', ''): os.path.join(base_output_dir, f) 
    for f in concat_file_list
}

view_videos = {
    'video_mix': concat_dict['video_mix'],  
    'plot': concat_dict['plot'],    
    'joint_Hips': concat_dict['joint_Hips'],
    'drum_dot': concat_dict['drum_dot'],
    'dance_dot': concat_dict['dance_dot'],
    
    **{f'{view}_view': concat_dict[f'{view}_view'] for view in views_to_generate}
}

### Generate Composite Video

In [ ]:
final_save_dir = os.path.dirname(base_output_dir)
process_layouts(layouts_to_export, view_videos, base_output_dir, file_name, dance_mode, mode_start_time, mode_end_time)